<a href="https://colab.research.google.com/github/gerryfrank10/NLP/blob/main/part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch

In [2]:
words = ['hello', 'world', 'i', 'am', 'gerald',
          'and', 'gerald', 'is', 'using', 'torch',
          'to', 'build', 'a', 'character', 'model',
          'also', 'giovanna', 'is', 'my', 'daughter',
          'and', 'she', 'is', 'very', 'cute', 'also',
          'my', 'wife', 'is', 'a', 'loving', 'person',
          'she','takes', 'care', 'of', 'me', 'and', 'our',
          'family','we', 'enjoy', 'spending', 'time', 'together',
          'for', 'some', 'reasons', 'i', 'need', 'to', 'query', 'this']

In [3]:
import string
chars = string.ascii_lowercase
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {i: ch for ch, i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [4]:
block_size = 3
X, Y = [], []

for w in words[:3]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X, dtype=torch.int64)
Y = torch.tensor(Y, dtype=torch.int64)

hello
... -> h
..h -> e
.he -> l
hel -> l
ell -> o
llo -> .
world
... -> w
..w -> o
.wo -> r
wor -> l
orl -> d
rld -> .
i
... -> i
..i -> .


In [5]:
import torch.nn.functional as F
import torch.nn as nn

In [6]:
C = torch.randn((27, 2))

In [7]:
C[5], C[5].dtype

(tensor([ 0.6472, -1.1181]), torch.float32)

In [8]:
# 1 , 27 x 27, 2 = 1, 2
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # We are typecasting to float to match the dtype of C

tensor([ 0.6472, -1.1181])

In [9]:
X.shape, Y.shape

(torch.Size([14, 3]), torch.Size([14]))

In [10]:
C[X]

tensor([[[-0.0096, -0.3447],
         [-0.0096, -0.3447],
         [-0.0096, -0.3447]],

        [[-0.0096, -0.3447],
         [-0.0096, -0.3447],
         [ 0.6169,  0.1482]],

        [[-0.0096, -0.3447],
         [ 0.6169,  0.1482],
         [ 0.6472, -1.1181]],

        [[ 0.6169,  0.1482],
         [ 0.6472, -1.1181],
         [ 0.0724, -1.9945]],

        [[ 0.6472, -1.1181],
         [ 0.0724, -1.9945],
         [ 0.0724, -1.9945]],

        [[ 0.0724, -1.9945],
         [ 0.0724, -1.9945],
         [ 1.7589,  0.8087]],

        [[-0.0096, -0.3447],
         [-0.0096, -0.3447],
         [-0.0096, -0.3447]],

        [[-0.0096, -0.3447],
         [-0.0096, -0.3447],
         [ 0.2100,  0.8273]],

        [[-0.0096, -0.3447],
         [ 0.2100,  0.8273],
         [ 1.7589,  0.8087]],

        [[ 0.2100,  0.8273],
         [ 1.7589,  0.8087],
         [-0.7421,  1.2301]],

        [[ 1.7589,  0.8087],
         [-0.7421,  1.2301],
         [ 0.0724, -1.9945]],

        [[-0.7421,  1

In [11]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  8],
        [ 0,  8,  5],
        [ 8,  5, 12],
        [ 5, 12, 12],
        [12, 12, 15],
        [ 0,  0,  0],
        [ 0,  0, 23],
        [ 0, 23, 15],
        [23, 15, 18],
        [15, 18, 12],
        [18, 12,  4],
        [ 0,  0,  0],
        [ 0,  0,  9]])

In [12]:
C[0], C[0], C[0], C[0], C[0], C[8]

(tensor([-0.0096, -0.3447]),
 tensor([-0.0096, -0.3447]),
 tensor([-0.0096, -0.3447]),
 tensor([-0.0096, -0.3447]),
 tensor([-0.0096, -0.3447]),
 tensor([0.6169, 0.1482]))

In [13]:
C[X].shape

torch.Size([14, 3, 2])

In [14]:
X[13, 2]

tensor(9)

In [15]:
C[X][13, 2]

tensor([ 0.0121, -1.3684])

In [16]:
C[9]

tensor([ 0.0121, -1.3684])

In [17]:
emb = C[X]  # (N, block_size, D)

In [18]:
w1 = torch.randn(size=(6, 100))
b1 = torch.randn(size=(100,))

In [19]:
# We can't do this emb @ w1 + b1 because the dimensions don't match
# We need to reshape emb to (N, block_size * D) to match w1
emb_reshaped = emb.view(emb.shape[0], -1)  # (N, block_size * D)
(emb_reshaped @ w1 + b1).shape  # (N, 100)

torch.Size([14, 100])

In [20]:
h = torch.tanh(emb_reshaped @ w1 + b1)  # (N, 100)
h.shape

torch.Size([14, 100])

In [21]:
W2 = torch.randn(size=(100, 27))
b2 = torch.randn(size=(27,))

In [22]:
logits = h @ W2 + b2  # (N, 27)

In [23]:
logits.shape

torch.Size([14, 27])

In [24]:
counts = logits.exp()

In [25]:
probs = counts / counts.sum(dim=1, keepdim=True)  # (N, 27)

In [26]:
probs[0].sum()

tensor(1.0000)

In [27]:
Y

tensor([ 8,  5, 12, 12, 15,  0, 23, 15, 18, 12,  4,  0,  9,  0])

In [28]:
probs.shape

torch.Size([14, 27])

In [29]:
probs[[0,1], Y[0]]  # Probability of the first character in the first word

tensor([7.4770e-10, 6.6980e-14])

In [30]:
probs[torch.arange(probs.shape[0]), Y] # probs[0].shape is 14 as with Y.shape = 14

tensor([7.4770e-10, 6.0842e-11, 3.2278e-05, 1.1914e-09, 9.8804e-08, 6.5523e-08,
        1.9243e-09, 3.0742e-12, 2.3060e-08, 1.3229e-06, 4.3239e-09, 4.5460e-09,
        1.6043e-06, 1.7268e-10])

In [31]:
loss = -probs[torch.arange(probs.shape[0]), Y].log().mean()
loss

tensor(18.5775)

In [36]:
g = torch.Generator().manual_seed(42)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn(size=(6,))


## TODO